# Phase 6C — validation optimization and F1 preparation

This notebook audits the private Phase 6B Epoch 3 checkpoint. It does not train a model, modify the frozen split, use the test set, or change the CER contract. It compares the existing one-clip-at-a-time generation path with batch sizes 4 and 8 on 12 deterministic validation clips.

Run the notebook cells (especially the subprocess cell) rather than pasting the CLI source into a code cell. The earlier `f_lineno` traceback was a secondary IPython formatting error after `argparse` rejected Jupyter's kernel arguments and missing required file paths. This wrapper supplies explicit paths and keeps those kernel arguments out of the validator.

Keep this notebook, its inputs and outputs private. The diagnostic output contains IDs, references and predictions.

## Private Kaggle inputs

Attach these private inputs with **Add Input**:

1. The existing Phase 5A validation ZIP containing all 732 validation clips.
2. `phase6b_P1_best_checkpoint_private.zip`, containing the best P1 model and `training_state.pt`.
3. A private code input containing `scripts/phase6c_validate.py`, or upload the BTP repository source as a private Kaggle Dataset.

Use a Tesla T4 GPU and the same pinned Transformers 4.57.1 / Tokenizers 0.22.2 / Hub 0.36.2 / Safetensors 0.8.0 stack. Internet is needed only if the processor files are not already cached.

In [ ]:
# Pin only the Hugging Face packages used by the audit; leave Kaggle PyTorch/CUDA unchanged.
import importlib.metadata as metadata
import subprocess, sys
PINS = {'transformers': '4.57.1', 'tokenizers': '0.22.2', 'huggingface-hub': '0.36.2', 'safetensors': '0.8.0'}
installed = {}
for package in PINS:
    try:
        installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed[package] = None
print('Installed:', installed)
if installed != PINS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', *[f'{k}=={v}' for k, v in PINS.items()]], check=True)
    raise RuntimeError('Packages were pinned. Restart the Kaggle session and run all cells again.')
try:
    import tqdm
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'tqdm'], check=True)
    raise RuntimeError('tqdm was installed. Restart the Kaggle session and run all cells again.')
print('tqdm:', tqdm.__version__)

In [ ]:
# Set these three paths after inspecting /kaggle/input. No test bundle belongs here.
from pathlib import Path
import subprocess, sys
input_root = Path('/kaggle/input')
# Kaggle can rename uploaded files, so inspect ZIP markers instead of assuming a filename.
import json, zipfile
all_zips = sorted(input_root.rglob('*.zip'))
def find_bundle(bundle_type):
    matches = []
    for archive in all_zips:
        try:
            with zipfile.ZipFile(archive) as zf:
                for name in zf.namelist():
                    if name.endswith('bundle_record.json'):
                        record = json.loads(zf.read(name))
                        if record.get('bundle_type') == bundle_type:
                            matches.append(archive)
        except (OSError, zipfile.BadZipFile, json.JSONDecodeError):
            pass
    return sorted(set(matches))
validation_candidates = find_bundle('phase5_validation_baseline')
if not validation_candidates:
    # Also accept an extracted private bundle directory mounted by Kaggle.
    for marker in input_root.rglob('bundle_record.json'):
        try:
            if json.loads(marker.read_text()).get('bundle_type') == 'phase5_validation_baseline':
                validation_candidates.append(marker.parent)
        except (OSError, json.JSONDecodeError):
            pass
checkpoint_candidates = [p for p in all_zips if 'best_checkpoint' in p.name.lower()]
if len(validation_candidates) != 1 or len(checkpoint_candidates) != 1:
    print('Mounted Kaggle ZIP files:')
    for p in all_zips: print(' -', p)
    raise FileNotFoundError('Attach exactly one private validation bundle and one best-checkpoint ZIP under Kaggle Add Input.')
validation_zip = validation_candidates[0]
checkpoint_zip = checkpoint_candidates[0]
code_candidates = list(input_root.rglob('phase6c_validate.py'))
if code_candidates:
    validator = code_candidates[0]
else:
    validator = Path('/kaggle/working/phase6c_validate.py')
    raise FileNotFoundError('Attach scripts/phase6c_validate.py as a private Kaggle code input.')
output_dir = Path('/kaggle/working/phase6c_private')
print({'validation_zip': validation_zip, 'checkpoint_zip': checkpoint_zip, 'validator': validator, 'output': output_dir})
assert 'test' not in validation_zip.name.lower() and 'test' not in checkpoint_zip.name.lower()

## What the audit records

The validator makes the effective Whisper prefix explicit: decoder start 50258, automatic language detection, transcribe token 50359, no-timestamps token, EOS 50257, greedy one-beam decoding, beginning suppression `[220, 50257]`, and maximum length 448. This preserves the Phase 6B runtime behavior while documenting the model-config fallback that caused the warning.

The validator stops with `mismatch` if batched token IDs, decoded text, normalized text or aggregate CER/S/D/I differ from the single-example path.

In [ ]:
# Run only the 12-clip equivalence audit. This does not train or score the test set.
# Live tqdm bars report archive checks, checksum verification, feature extraction,
# single-clip generation, each tested batch size, elapsed time, and GPU memory.
command = [sys.executable, str(validator), '--validation-zip', str(validation_zip), '--checkpoint', str(checkpoint_zip), '--output', str(output_dir), '--batch-sizes', '4', '8']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Read the aggregate audit result. Keep the diagnostic JSONL private.
import json
equivalence = json.loads((output_dir / 'phase6c_equivalence.json').read_text())
print('Status:', equivalence['status'])
print('Chosen validation batch size:', equivalence.get('chosen_batch_size'))
print('Single:', equivalence['single'])
for result in equivalence['batched']:
    print('Batched:', result)
assert equivalence['status'] == 'PASS', 'Do not proceed to F1 until exact equivalence passes.'

## Decision gate

Proceed to F1 only if `phase6c_equivalence.json` reports `PASS` and at least one batch size has exact equality. Use the largest passing batch size for validation. If the result is `mismatch` or both batches fail, stop and investigate before F1.

F1 must start from the original pinned pretrained checkpoint on all 3,629 training clips. Do not warm-start it from the P1 checkpoint. Keep all 683 test clips sealed.